In [ ]:
import duckdb

In [ ]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [ ]:
df = con.execute("""
                 select *
                 from (
                    select *, row_number() over (partition by NATBR order by data_ingestao desc) as row
                    from bronze_z0019
                 where data_ingestao >= '2026-05-12'
                 ) where row = 1
                 """).fetchdf()
df.head(10)

In [ ]:
df_final = df.drop(columns=["nome_arquivo","data_ingestao","row"])
df_final = df_final.rename(columns={"NATBR":"id"})
df_final = df_final.rename(columns={"MAKTX":"nm_produto"})
df_final = df_final.rename(columns={"WERKS":"id_categoria"})
df_final = df_final.rename(columns={"MAINS":"id_fornecedor"})
df_final = df_final.rename(columns={"LABST":"vl_preco"})
df_final.head(10)


In [ ]:
df_final.dtypes

In [ ]:
df2 = df_final
df2 = df2.astype(
    {
        "id": "int",
        "nm_produto": "str",
        "id_categoria": "str",
        "id_fornecedor": "int",
        "vl_preco": "float"
    }
)

In [ ]:
df2.dtypes

In [ ]:
con.execute("""
            create table if not exists produtos (
            id BIGINT,
            nm_produto text,
            id_categoria text,
            id_fornecedor BIGINT,
            vl_preco FLOAT
            )
            """)


In [ ]:
con.execute("insert into produtos select * from df2")

In [ ]:
df.resultado = con.execute("select * from produtos").fetchdf()
df.resultado.head(10)

In [ ]:
con.close()